In [24]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import ssl
from torch.utils.data import random_split
ssl._create_default_https_context = ssl._create_unverified_context

In [ ]:
BATCH_SIZE = 64
IMG_NORM_MEAN = 0.5
IMG_NORM_STD = 0.5
NUM_EPOCHS = 1000
time_stamps
BETAS = torch.linspace(0.0001,0.02,)
ALPHAS = 1-BETAS
ALPHA_Ts = torch.cumprod(ALPHAS,dim=0)
NUM_CHANNELS = 3 
IMG_HEIGHT = 32
IMG_WIDTH = 32

In [164]:
class Denoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.time_embed = nn.Embedding(1000, 64)
        self.conv1 = nn.Conv2d(3,  64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 3,  kernel_size=3, padding=1)

    def forward(self, noisy_images, time_stamps):
        t_emb = self.time_embed(time_stamps)        # (B, 64)
        t_emb = t_emb.view(-1, 64, 1, 1)          # (B, 64, 1, 1)
        x = torch.relu(self.conv1(noisy_images))    # (B, 64, 32, 32)
        x = x + t_emb                              
        x = torch.relu(self.conv2(x))
        x = self.conv3(x)
        return x                                        # (B,3,32,32)

In [26]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((IMG_NORM_MEAN,IMG_NORM_MEAN,IMG_NORM_MEAN),
                         (IMG_NORM_STD,IMG_NORM_STD,IMG_NORM_STD))
])
train_val_dataset = datasets.CIFAR10(root="./train_val",download=True,train=True,transform=transform)
test_dataset = datasets.CIFAR10(root="./test",download=True,train=False,transform=transform)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [27]:
train_size = int(0.8 * len(train_val_dataset))
val_size = len(train_val_dataset) - train_size

train_dataset, val_dataset = random_split(train_val_dataset, [train_size, val_size])

In [28]:
train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=True)

In [ ]:
denoiser = Denoiser()
optimizer = torch.optim.Adam(denoiser.parameters(),lr=1e-3)
loss_fn =  nn.MSELoss()

In [ ]:
for epoch in range(NUM_EPOCHS):
    train_loss = 0.0
    val_loss = 0.0
    test_loss = 0.0
    denoiser.train()
    for images,_ in train_loader:
        sampled_time_stamps = torch.randint(1,1000,size=(len(images),))
        random_error_sampled = torch.randn(size=(BATCH_SIZE,NUM_CHANNELS,IMG_HEIGHT,IMG_WIDTH))
        alphats_time_based = ALPHA_Ts[sampled_time_stamps].view(-1,1,1,1)
        new_images = torch.sqrt(alphats_time_based) * images + torch.sqrt(1-alphats_time_based) * random_error_sampled
        predicted_noise = denoiser.forward(new_images,sampled_time_stamps)
        loss = loss_fn(predicted_noise,random_error_sampled)
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    with torch.no_grad():
        denoiser.eval()
        for images,_ in val_loader:
            sampled_time_stamps = torch.randint(1,1000,size=(len(images),))
            random_error_sampled = torch.randn(size=(BATCH_SIZE,NUM_CHANNELS,IMG_HEIGHT,IMG_WIDTH))
            alphats_time_based = ALPHA_Ts[sampled_time_stamps].view(-1,1,1,1)
            new_images = torch.sqrt(alphats_time_based) * images + torch.sqrt(1-alphats_time_based) * random_error_sampled
            predicted_noise = denoiser.forward(new_images,sampled_time_stamps)
            loss = loss_fn(predicted_noise,random_error_sampled)
            val_loss += loss.item()
        for images,_ in test_loader:
            sampled_time_stamps = torch.randint(1,1000,size=(len(images),))
            random_error_sampled = torch.randn(size=(BATCH_SIZE,NUM_CHANNELS,IMG_HEIGHT,IMG_WIDTH))
            alphats_time_based = ALPHA_Ts[sampled_time_stamps].view(-1,1,1,1)
            new_images = torch.sqrt(alphats_time_based) * images + torch.sqrt(1-alphats_time_based) * random_error_sampled
            predicted_noise = denoiser.forward(new_images,sampled_time_stamps)
            loss = loss_fn(predicted_noise,random_error_sampled)
            test_loss += loss.item()
    train_loss = train_loss/len(train_loader)
    val_loss = val_loss/len(val_loader)
    test_loss = test_loss/len(test_loader)
    print(f"Epoch {epoch+1} Train Loss = {train_loss:.6f} Val Loss = {val_loss:.6f} Test Loss = {test_loss:.6f}")

       



In [112]:
torch.randint(size=(32,),low=1,high=1001)

tensor([682, 960, 665, 541, 527, 401, 878, 310, 182, 702, 641, 440, 564, 112,
         82, 390,  26, 478, 984, 528,  17,  62, 549, 260, 622, 165, 939,  82,
        867,   4, 107, 200])

In [155]:
torch.randn(size=(10,))

tensor([-0.0063, -0.4589,  0.4030, -0.7579, -0.8935, -1.5210,  0.7558, -0.5696,
         1.3900,  1.4574])